# 语言模型张量与 Loss

这个 Notebook 用一句短文本跑通从 tokenizer 输出到 next-token cross entropy 的完整数据流。

In [1]:
import warnings

warnings.filterwarnings("ignore", message="IProgress not found.*")

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer

# 固定随机种子，使随机 logits 可以复现。
torch.manual_seed(42)

## 1. 编码一句中文

这里只加载 Qwen tokenizer，不加载模型权重。Qwen 使用 byte-level BPE，因此同时观察词表内部表示和解码后的人类可读片段。

In [2]:
# 编码文本，并同时观察内部 token 和可读解码。
model_id = "Qwen/Qwen3-0.6B-Base"
tokenizer = AutoTokenizer.from_pretrained(model_id)

text = "今天天气很好。"
token_ids = tokenizer(
    text, add_special_tokens=False, return_tensors="pt"
)["input_ids"]
internal_tokens = tokenizer.convert_ids_to_tokens(token_ids[0])
readable_tokens = [
    tokenizer.decode([token_id]) for token_id in token_ids[0].tolist()
]
decoded_text = tokenizer.decode(token_ids[0])

print("text:", text)
print("internal tokens:", internal_tokens)
print("readable tokens:", readable_tokens)
print("token IDs:", token_ids)
print("token_ids shape:", tuple(token_ids.shape))
print("decoded text:", decoded_text)
print("round trip:", decoded_text == text)

text: 今天天气很好。
internal tokens: ['ä»Ĭå¤©', 'å¤©æ°Ķ', 'å¾Īå¥½', 'ãĢĤ']
readable tokens: ['今天', '天气', '很好', '。']
token IDs: tensor([[100644, 104307, 101243,   1773]])
token_ids shape: (1, 4)
decoded text: 今天天气很好。
round trip: True


## 2. 手动构造 shifted labels

对于 token 序列 $[x_0, x_1, \ldots, x_T]$，模型读取前 $T$ 个 token，并分别预测紧随其后的 $T$ 个 token。

In [3]:
# 输入去掉末尾，标签去掉开头，形成相邻 token 预测对。
input_ids = token_ids[:, :-1]
labels = token_ids[:, 1:]

print("input_ids:", input_ids)
print("labels:   ", labels)
print("input_ids shape:", tuple(input_ids.shape))
print("labels shape:   ", tuple(labels.shape))

input_ids: tensor([[100644, 104307, 101243]])
labels:    tensor([[104307, 101243,   1773]])
input_ids shape: (1, 3)
labels shape:    (1, 3)


## 3. 构造 logits 并计算交叉熵

序列中的每个位置都需要为词表里的每个 token 产生一个分数，因此 logits 的形状是 `[B, T, V]`。labels 在每个位置只保存正确 token 的 ID，所以形状是 `[B, T]`。

In [4]:
# 为每个位置随机生成覆盖全部词表的预测分数。
batch_size, sequence_length = labels.shape
vocab_size = len(tokenizer)
logits = torch.randn(batch_size, sequence_length, vocab_size)

# 展平 B 和 T，让交叉熵一次处理所有位置。
loss = F.cross_entropy(
    logits.reshape(-1, vocab_size),
    labels.reshape(-1),
)

print("logits shape:", tuple(logits.shape))
print("labels shape: ", tuple(labels.shape))
print("loss:", loss.item())

logits shape: (1, 3, 151669)
labels shape:  (1, 3)
loss: 12.945048332214355


## 4. 在 loss 中忽略 padding

前面只有一句文本。这里一次编码两句话，`B = 2`。因为两句话的 token 数不同，需要先用 padding 把它们补成相同长度。

### 4.1 编码两句话并观察 padding

`input_ids` 保存 token ID，`attention_mask` 用 `1` 标记真实 token、用 `0` 标记 padding。

In [5]:
# 批量编码时将较短文本补齐到同一长度。
texts = ["今天天气很好。", "你好。"]
batch = tokenizer(
    texts, add_special_tokens=False, padding=True, return_tensors="pt"
)

batch_token_ids = batch["input_ids"]
batch_attention_mask = batch["attention_mask"]

print("pad token:", tokenizer.pad_token)
print("pad token ID:", tokenizer.pad_token_id)
print("input_ids shape:", tuple(batch_token_ids.shape))
print("input_ids:\n", batch_token_ids)
print("attention_mask:\n", batch_attention_mask)

pad token: <|endoftext|>
pad token ID: 151643
input_ids shape: (2, 4)
input_ids:
 tensor([[100644, 104307, 101243,   1773],
        [108386,   1773, 151643, 151643]])
attention_mask:
 tensor([[1, 1, 1, 1],
        [1, 1, 0, 0]])


第二句话更短，所以末尾出现 padding。padding 在 `input_ids` 中仍有一个整数 ID，必须结合 `attention_mask` 才知道哪些位置只是补齐。

### 4.2 同时 shift 输入、标签和 mask

输入删除最后一个 token，标签删除第一个 token。标签对应的 mask 也必须删除第一个位置，才能继续与标签对齐。

In [6]:
# 输入、标签和 mask 使用相同的单 token 错位。
batch_inputs = batch_token_ids[:, :-1]
batch_labels = batch_token_ids[:, 1:].clone()
label_mask = batch_attention_mask[:, 1:]

print("inputs:\n", batch_inputs)
print("labels before masking:\n", batch_labels)
print("label mask:\n", label_mask)
print("all shapes:", batch_inputs.shape, batch_labels.shape, label_mask.shape)

inputs:
 tensor([[100644, 104307, 101243],
        [108386,   1773, 151643]])
labels before masking:
 tensor([[104307, 101243,   1773],
        [  1773, 151643, 151643]])
label mask:
 tensor([[1, 1, 1],
        [1, 0, 0]])
all shapes: torch.Size([2, 3]) torch.Size([2, 3]) torch.Size([2, 3])


### 4.3 把 padding 标签替换为 `-100`

`label_mask == 0` 的位置对应 padding，不参与训练目标。将这些 labels 改成 `-100`，因为 `F.cross_entropy` 默认忽略 `-100`。

In [7]:
# 用 -100 标出不计入 loss 的 padding 标签。
masked_labels = batch_labels.clone()
masked_labels[label_mask == 0] = -100

print("labels before masking:\n", batch_labels)
print("labels after masking:\n", masked_labels)

labels before masking:
 tensor([[104307, 101243,   1773],
        [  1773, 151643, 151643]])
labels after masking:
 tensor([[104307, 101243,   1773],
        [  1773,   -100,   -100]])


### 4.4 计算 batch loss

现在每个有效位置都有一个正确 token ID，每个 padding 位置都是 `-100`。交叉熵只对有效位置求平均。

In [8]:
# 交叉熵只统计标签不为 -100 的位置。
batch_size, sequence_length = masked_labels.shape
batch_logits = torch.randn(batch_size, sequence_length, vocab_size)

flat_logits = batch_logits.reshape(-1, vocab_size)
flat_labels = masked_labels.reshape(-1)

batch_loss = F.cross_entropy(
    flat_logits,
    flat_labels,
    ignore_index=-100,
)

print("logits shape before flatten:", tuple(batch_logits.shape))
print("logits shape after flatten: ", tuple(flat_logits.shape))
print("labels shape after flatten: ", tuple(flat_labels.shape))
print("valid target count:", (flat_labels != -100).sum().item())
print("ignored target count:", (flat_labels == -100).sum().item())
print("batch loss:", batch_loss.item())

logits shape before flatten: (2, 3, 151669)
logits shape after flatten:  (6, 151669)
labels shape after flatten:  (6,)
valid target count: 4
ignored target count: 2
batch loss: 12.861200332641602


## 5. 理解 causal mask

Causal mask 规定当前位置在计算 attention 时能够读取的序列位置。

### 5.1 先明确每个位置正在预测什么

这里复用第一句话 shift 后的三个输入位置。每一行是一个 query，也就是当前正在计算表示的输入位置；每一列是一个 key，也就是它可能读取的位置。

In [9]:
# 逐位置查看输入 token 与下一个目标 token。
input_tokens = readable_tokens[:-1]
target_tokens = readable_tokens[1:]
causal_sequence_length = len(input_tokens)

for position, (input_token, target_token) in enumerate(
    zip(input_tokens, target_tokens)
):
    print(f"位置 {position}: 输入 {input_token!r}，预测 {target_token!r}")

位置 0: 输入 '今天'，预测 '天气'
位置 1: 输入 '天气'，预测 '很好'
位置 2: 输入 '很好'，预测 '。'


### 5.2 构造禁止读取未来的矩阵

矩阵中 `True` 表示禁止读取。行号是当前 query 位置，列号是被读取的 key 位置。例如 `[0, 1] = True` 表示位置 0 不能读取未来的位置 1。

In [10]:
# 上三角区域对应每个 query 右侧的未来 key。
causal_mask = torch.triu(
    torch.ones(
        causal_sequence_length, causal_sequence_length, dtype=torch.bool
    ),
    diagonal=1,
)

print("列（key）token:", input_tokens)
print("行（query）token:", input_tokens)
print("causal mask（1=True=禁止读取，0=False=允许读取）:\n", causal_mask.int())

列（key）token: ['今天', '天气', '很好']
行（query）token: ['今天', '天气', '很好']
causal mask（1=True=禁止读取，0=False=允许读取）:
 tensor([[0, 1, 1],
        [0, 0, 1],
        [0, 0, 0]], dtype=torch.int32)


逐行读取这个矩阵：

- 行 0 只能读取位置 0；
- 行 1 可以读取位置 0、1；
- 行 2 可以读取位置 0、1、2。

对角线允许读取，因为位置 $t$ 使用当前输入 token $x_t$ 预测下一个 token $x_{t+1}$。读取 $x_t$ 属于正常的当前信息。

### 5.3 把 mask 应用到 attention scores

mask 会把所有未来位置的 score 改成负无穷。经过 softmax 后，这些位置的 attention weight 就严格等于 0。

In [11]:
# 先屏蔽未来分数，再沿每行的 key 维度归一化。
attention_scores = torch.tensor([
    [1.0, 2.0, 3.0],
    [1.0, 2.0, 3.0],
    [1.0, 2.0, 3.0],
])
masked_scores = attention_scores.masked_fill(causal_mask, float("-inf"))
attention_weights = F.softmax(masked_scores, dim=-1)

print("原始 attention scores:\n", attention_scores)
print("mask 后的 scores:\n", masked_scores)
print("softmax 后的 weights:\n", attention_weights)

原始 attention scores:
 tensor([[1., 2., 3.],
        [1., 2., 3.],
        [1., 2., 3.]])
mask 后的 scores:
 tensor([[1., -inf, -inf],
        [1., 2., -inf],
        [1., 2., 3.]])
softmax 后的 weights:
 tensor([[1.0000, 0.0000, 0.0000],
        [0.2689, 0.7311, 0.0000],
        [0.0900, 0.2447, 0.6652]])


### 5.4 不要和 padding mask 混淆

- `causal_mask [T,T]`：根据位置关系屏蔽未来 token；
- `attention_mask [B,T]`：根据每条样本的实际长度屏蔽 padding；
- `labels == -100 [B,T]`：让相应位置不参与 loss。

Decoder-only 模型会在内部把 causal mask 与 padding mask 组合起来。允许读取的位置需要同时满足两个条件：位于当前或过去，并且属于真实 token。

In [12]:
# 直接检查所有被屏蔽位置的 attention weight。
future_weights = attention_weights.masked_select(causal_mask)
print("所有未来位置的 attention weights:", future_weights)
print("未来位置是否全部为 0:", torch.all(future_weights == 0).item())

所有未来位置的 attention weights: tensor([0., 0., 0.])
未来位置是否全部为 0: True


这个检查确认：在当前示例中，所有未来位置的 attention weight 都是 0。

## 6. 训练流程中的必备概念

前五节解释了数据如何从 token 变成 loss。本节只建立后续训练需要的概念，不实现完整训练循环。

### 6.1 Embedding

Token ID 只是词表索引，ID 的数值大小没有语义。Embedding 是一张形状为 `[V,D]` 的可训练查找表：每个 token ID 取出一行长度为 `D` 的向量。

```text
token IDs [B,T]
→ embedding lookup [V,D]
→ token vectors [B,T,D]
```

In [13]:
# Embedding 按 token ID 查表，得到每个位置的向量。
embedding_dim = 8
embedding = torch.nn.Embedding(
    num_embeddings=vocab_size,
    embedding_dim=embedding_dim,
)
token_vectors = embedding(input_ids)

print("input_ids shape:", tuple(input_ids.shape))
print("embedding table shape:", tuple(embedding.weight.shape))
print("token vectors shape:", tuple(token_vectors.shape))

input_ids shape: (1, 3)
embedding table shape: (151669, 8)
token vectors shape: (1, 3, 8)


### 6.2 Teacher forcing

训练时，模型每个位置看到的历史输入全部来自真实文本。模型上一步的生成结果不会作为下一个训练位置的输入。当前 Notebook 的 shifted inputs 和 labels 就体现了 teacher forcing：

```text
真实序列：今天  天气  很好  。
模型输入：今天  天气  很好
训练目标：天气  很好  。
```

Teacher forcing 一次提供完整的 shifted input。Causal mask 限制每个位置的可见范围，因此训练时可以同时计算所有位置的 loss。

生成时没有后续的真实 token。模型需要把刚生成的 token 接回输入，再生成下一个 token。

这里的 teacher 指训练数据提供的真实 token，与 Teacher 模型无关。

### 6.3 Gradient accumulation

Gradient accumulation 用于显存放不下大 batch 的情况。它把一个大 batch 拆成多个较小的 micro-batch，依次计算后再统一更新一次模型参数：

```text
多个 micro-batch
→ 梯度依次累加
→ 累加完成后更新一次参数
```

阶段 0 只需记住它的用途。梯度如何累积、参数如何更新，以及 loss 为什么需要缩放，会在阶段 1 学习 `backward()` 和 optimizer 后再展开。

### 6.4 Train / validation split

数据必须分成互不重叠的训练集和验证集：

| 数据 | 用途 | 是否反向传播 | 是否更新参数 |
| --- | --- | --- | --- |
| Train | 学习模型参数 | 是 | 是 |
| Validation | 检查模型在未参与训练的数据上的表现 | 否 | 否 |

训练 loss 下降而 validation loss 上升，通常意味着过拟合。验证阶段通常使用 `model.eval()` 和 `torch.no_grad()`，并且不能根据 validation 样本继续执行参数更新。

### 6.5 Checkpoint

Checkpoint 是可以恢复训练状态的快照，不应只保存模型权重。最小内容通常包括：

```python
checkpoint = {
    "model": model.state_dict(),
    "optimizer": optimizer.state_dict(),
    "step": step,
    "config": config,
}
```

模型状态保存当前参数值。`optimizer` 状态保存动量等信息。`step` 和 `config` 记录恢复位置与训练设置。只保存模型权重通常可以推理，但不一定能无缝恢复训练。阶段 1 会实际完成 checkpoint 保存与恢复。

### 6.6 把概念串起来

```text
Train tokens
→ Embedding
→ Causal attention
→ Logits
→ Teacher-forced labels
→ Cross entropy
→ Gradient accumulation
→ Optimizer step
→ Checkpoint

Validation tokens
→ 同一模型 forward
→ 只计算 validation loss，不更新参数
```

## 验收问题

### 1. labels 为什么要 shift？

> 因为每个输入位置的训练目标，都是预测紧随其后的下一个 token。

### 2. logits 为什么多一个词表维度？

> 因为模型必须在每个序列位置上，给所有可能的下一个 token 分别打分。

### 3. padding 为什么要忽略？

> padding 只是 batch 对齐手段，不属于真实语言数据；让它参与 loss 会引入错误的学习目标。

### 4. causal mask 防止什么信息泄漏？

> 它阻止当前位置在预测下一个 token 时直接读取未来的真实 token。